# Variational Inference — Exercises

Companion to the note [Variational Inference](Variational%20Inference.md).

Practise the ELBO and its gap, KL divergences (and why $D_{KL}(q\Vert p)$ is mode seeking), mean-field coordinate ascent (CAVI), Monte Carlo ELBO estimates, the reparameterization trick vs score-function gradients, black-box VI, and the link to EM, by hand and from scratch in NumPy, checked against exact posteriors.

**15 exercises** · 🧠 Concept ×4 · ✍️ By hand ×5 · 💻 Code ×6

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from scipy import stats
from scipy.integrate import quad
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)
# Shared conjugate model: mu ~ N(0, 10^2), x_i | mu ~ N(mu, 1), 50 observations
x_vi = rng.normal(2.0, 1.0, 50)
N_vi, prior_var = len(x_vi), 100.0
post_var_exact = 1 / (1 / prior_var + N_vi)
post_mean_exact = post_var_exact * x_vi.sum()

## Part A · Concepts

### Exercise 1 · Why maximize the ELBO?
*🧠 Concept · ★☆☆*

We want $q$ close to $p(z\mid x)$, but we cannot evaluate $p(z\mid x)$. Explain why maximizing the ELBO
$\mathbb E_q[\log p(x,z)]-\mathbb E_q[\log q(z)]$ is equivalent to minimizing $D_{KL}(q\Vert p(z\mid x))$ and why the ELBO is computable.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Use the identity $\log p(x)=\text{ELBO}(q)+D_{KL}(q\Vert p(z\mid x))$ (Exercise 5).

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\log p(x)$ does not depend on $q$, so increasing the ELBO by some amount decreases the KL by exactly the same amount.
The ELBO only needs the **joint** $p(x,z)=p(x\mid z)p(z)$, which we can evaluate pointwise, and expectations under our own $q$, which we can compute analytically or by sampling from $q$. The intractable normalizer $p(x)$ never appears.
Bonus: the ELBO is a lower bound on the log evidence, usable (with care) for model comparison.

</details>

### Exercise 2 · Mode seeking vs mass covering
*🧠 Concept · ★☆☆*

VI minimizes the **reverse** KL $D_{KL}(q\Vert p)=\mathbb E_q[\log q-\log p]$. Expectation propagation and MLE-style fitting minimize the **forward**
KL $D_{KL}(p\Vert q)$. If $p$ is bimodal and $q$ is a single Gaussian, sketch what each solution looks like and explain using where each KL is infinite.
Connect this to the note's statement that VI "under-estimates posterior variance".

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Reverse KL: $q\log(q/p)$ blows up where $q>0$ but $p\approx0$. Forward KL: $p\log(p/q)$ blows up where $p>0$ but $q\approx0$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

**Reverse KL** (VI) penalizes putting $q$-mass where $p$ is tiny ("zero-forcing"), so $q$ locks onto **one mode** and is narrow: mode seeking.
**Forward KL** penalizes missing any region where $p$ has mass ("zero-avoiding"), so $q$ stretches over **both modes**, with its mean in the valley between them: mass covering (for Gaussian $q$ it is exact moment matching).
The same zero-forcing makes mean-field VI too narrow even for unimodal correlated posteriors: it cannot represent the correlation, and the reverse KL prefers to shrink rather than spill outside $p$ (Exercises 10 and 11).

</details>

### Exercise 3 · EM is VI with an exact E-step
*🧠 Concept · ★★☆*

The note says "EM is VI with $q=p(z\mid x,\theta)$". Describe EM as coordinate ascent on $\mathcal L(q,\theta)$, say what happens to the gap in the E-step,
and explain what **variational EM** does when $p(z\mid x,\theta)$ is intractable (e.g. LDA). See [[Gaussian Mixture Models and EM]].

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Two blocks of variables: $q$ and $\theta$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

E-step: maximize $\mathcal L$ over $q$ with $\theta$ fixed. Without restrictions the optimum is $q=p(z\mid x,\theta)$, making the gap $D_{KL}=0$, so the bound touches $\log p(x\mid\theta)$.
M-step: maximize $\mathcal L$ over $\theta$ with $q$ fixed (the expected complete-data log-likelihood). Each step increases $\mathcal L$, giving monotone EM.
Variational EM restricts $q$ to a tractable family (e.g. mean field) in the E-step. The gap no longer closes, so we maximize a lower bound on the likelihood rather than the likelihood itself; the result is an approximation, but each step still increases the ELBO.

</details>

### Exercise 4 · What mean-field throws away
*🧠 Concept · ★★☆*

Mean-field assumes $q(z)=\prod_iq_i(z_i)$. Give two concrete consequences for a posterior where $z_1,z_2$ are strongly correlated (e.g. slope and intercept
of a regression with uncentred inputs). How can you reduce the problem?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

What is $\mathrm{Cov}_q(z_1,z_2)$ under a factorized $q$?

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $\mathrm{Cov}_q(z_1,z_2)=0$ by construction: any joint statement ("if the slope is high, the intercept is low") is lost, and so are credible regions for functions of both.
2. The marginal variances are **underestimated**: for a Gaussian target the CAVI solution has variances $1/\Lambda_{ii}$ (conditional variances) instead of $\Sigma_{ii}$ (Exercise 10), very overconfident when correlation is strong.
Fixes: reparameterize to reduce correlation (centre the inputs), use structured / full-rank Gaussian VI, normalizing flows, or [[MCMC]].

</details>

## Part B · By hand

### Exercise 5 · The ELBO identity
*✍️ By hand · ★★☆*

Prove $\log p(x)=\underbrace{\mathbb E_q[\log p(x,z)]-\mathbb E_q[\log q(z)]}_{\text{ELBO}}+D_{KL}(q(z)\Vert p(z\mid x))$, and the note's second form
$\text{ELBO}=\mathbb E_q[\log p(x\mid z)]-D_{KL}(q(z)\Vert p(z))$. Which form is used in a VAE, and what do its two terms mean?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Write $\log p(x)=\log p(x,z)-\log p(z\mid x)$, which holds for every $z$, then take $\mathbb E_q$ after adding and subtracting $\log q(z)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

For any $z$: $\log p(x)=\log\frac{p(x,z)}{q(z)}+\log\frac{q(z)}{p(z\mid x)}$. Taking $\mathbb E_q$ (the left side is constant) gives ELBO $+\,D_{KL}(q\Vert p(z\mid x))$; since KL $\ge0$, ELBO $\le\log p(x)$.
Second form: $\log p(x,z)=\log p(x\mid z)+\log p(z)$, so ELBO $=\mathbb E_q[\log p(x\mid z)]-\mathbb E_q[\log q(z)-\log p(z)]$.
VAEs use the second form ([[Generative Models]]): a **reconstruction** term (decoder likelihood) and a **regularizer** keeping the encoder's $q(z\mid x)$ close to the prior, often available in closed form (Exercise 6).

</details>

### Exercise 6 · KL between two Gaussians
*✍️ By hand · ★☆☆*

Use $D_{KL}(\mathcal N(\mu_1,\sigma_1^2)\Vert\mathcal N(\mu_2,\sigma_2^2))=\log\frac{\sigma_2}{\sigma_1}+\frac{\sigma_1^2+(\mu_1-\mu_2)^2}{2\sigma_2^2}-\frac12$ to compute
$D_{KL}(\mathcal N(1,1)\Vert\mathcal N(0,4))$ and the reverse $D_{KL}(\mathcal N(0,4)\Vert\mathcal N(1,1))$. (Here $4$ is the variance.)

In [ ]:
kl_qp = None
kl_pq = None

_kl = lambda a, b: quad(lambda z: a.pdf(z) * (a.logpdf(z) - b.logpdf(z)), -30, 30)[0]
check('KL(N(1,1) || N(0,4))', kl_qp, _kl(stats.norm(1, 1), stats.norm(0, 2)), tol=1e-6)
check('KL(N(0,4) || N(1,1))', kl_pq, _kl(stats.norm(0, 2), stats.norm(1, 1)), tol=1e-6)

<details>
<summary><b>💡 Hint</b></summary>

$\sigma_2=2$ in the first case. $\ln2\approx0.693$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$D_{KL}(\mathcal N(1,1)\Vert\mathcal N(0,4))=\ln2+\frac{1+1}{8}-\frac12\approx0.443$.
$D_{KL}(\mathcal N(0,4)\Vert\mathcal N(1,1))=\ln\frac12+\frac{4+1}{2}-\frac12=2-\ln2\approx1.307$.
KL is **not symmetric**: a wide distribution approximated by a narrow one is penalized much more.

```python
kl_qp = np.log(2) + 2 / 8 - 0.5
kl_pq = -np.log(2) + 5 / 2 - 0.5
```

</details>

### Exercise 7 · ELBO and gap in a two-state model
*✍️ By hand · ★★☆*

Latent $z\in\{0,1\}$ with $p(z=1)=0.5$; one observation $x$ with $p(x\mid z=1)=0.8$, $p(x\mid z=0)=0.2$.
(a) Compute $\log p(x)$ and the posterior $p(z=1\mid x)$. (b) For $q(z=1)=0.5$ compute the ELBO and the gap. (c) Which $q$ maximizes the ELBO?

In [ ]:
log_px = None
post_z1 = None
elbo_half = None
gap_half = None

_pj = np.array([0.5 * 0.2, 0.5 * 0.8]); _px = _pj.sum(); _q = np.array([.5, .5])
check('log p(x)', log_px, np.log(_px)); check('p(z=1|x)', post_z1, _pj[1] / _px)
check('ELBO for q=0.5', elbo_half, np.sum(_q * (np.log(_pj) - np.log(_q))))
check('gap = KL(q || posterior)', gap_half, stats.entropy(_q, _pj / _px))

<details>
<summary><b>💡 Hint</b></summary>

$p(x,z=1)=0.4$, $p(x,z=0)=0.1$. ELBO $=\sum_zq(z)[\log p(x,z)-\log q(z)]$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $p(x)=0.5$, $\log p(x)\approx-0.693$; $p(z=1\mid x)=0.4/0.5=0.8$.
(b) ELBO $=0.5(\log0.4-\log0.5)+0.5(\log0.1-\log0.5)=0.5(-0.223)+0.5(-1.609)\approx-0.916$; gap $=-0.693+0.916\approx0.223=D_{KL}(q\Vert p(z\mid x))$.
(c) $q(z=1)=0.8$, the exact posterior: then ELBO $=\log p(x)$ and the gap is 0.

```python
log_px = np.log(0.5)
post_z1 = 0.8
elbo_half = 0.5 * (np.log(0.4) - np.log(0.5)) + 0.5 * (np.log(0.1) - np.log(0.5))
gap_half = log_px - elbo_half
```

</details>

### Exercise 8 · The reparameterization trick by hand
*✍️ By hand · ★☆☆*

Let $q=\mathcal N(\mu,\sigma^2)$ and $f(z)=z^2$. Write $z=\mu+\sigma\varepsilon$, $\varepsilon\sim\mathcal N(0,1)$ and compute
$\nabla_\mu\mathbb E_q[f(z)]$ and $\nabla_\sigma\mathbb E_q[f(z)]$ by moving the gradient inside the expectation. Evaluate at $\mu=1.5$, $\sigma=0.5$.

In [ ]:
grad_mu = None
grad_sigma = None

_E = lambda m, s: quad(lambda z: z ** 2 * stats.norm.pdf(z, m, s), -20, 20)[0]
_h = 1e-5
check('d/dmu (finite differences)', grad_mu, (_E(1.5 + _h, .5) - _E(1.5 - _h, .5)) / (2 * _h), tol=1e-4)
check('d/dsigma (finite differences)', grad_sigma, (_E(1.5, .5 + _h) - _E(1.5, .5 - _h)) / (2 * _h), tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

$\nabla_\mu f(\mu+\sigma\varepsilon)=2(\mu+\sigma\varepsilon)$ and $\nabla_\sigma f=2(\mu+\sigma\varepsilon)\varepsilon$. Use $\mathbb E[\varepsilon]=0$, $\mathbb E[\varepsilon^2]=1$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\nabla_\mu=\mathbb E[2(\mu+\sigma\varepsilon)]=2\mu=3$; $\nabla_\sigma=\mathbb E[2(\mu+\sigma\varepsilon)\varepsilon]=2\sigma=1$.
Check: $\mathbb E[z^2]=\mu^2+\sigma^2$ has exactly these derivatives. The trick moves the randomness into a parameter-free $\varepsilon$, so gradients flow through samples (the basis of VAEs and BBVI, Exercise 13).

```python
grad_mu = 3.0
grad_sigma = 1.0
```

</details>

### Exercise 9 · Deriving the mean-field update for a Gaussian
*✍️ By hand · ★★★*

Target $p(z)=\mathcal N(z\mid\mu,\Lambda^{-1})$ in 2-D. With $q=q_1(z_1)q_2(z_2)$ the CAVI update is $\log q_1^*(z_1)=\mathbb E_{q_2}[\log p(z)]+\text{const}$.
Show that $q_1^*=\mathcal N\big(m_1,\Lambda_{11}^{-1}\big)$ with $m_1=\mu_1-\Lambda_{11}^{-1}\Lambda_{12}(\mathbb E[z_2]-\mu_2)$.
At the fixed point, what are $m_1,m_2$? Why are the variances too small?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Expand $-\tfrac12(z-\mu)^\top\Lambda(z-\mu)$ and keep only the terms with $z_1$; they are quadratic in $z_1$, so $q_1$ is Gaussian.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Terms with $z_1$: $-\tfrac12\Lambda_{11}z_1^2+z_1\mu_1\Lambda_{11}-z_1\Lambda_{12}(\mathbb E[z_2]-\mu_2)$. Completing the square gives precision $\Lambda_{11}$ and mean $m_1=\mu_1-\Lambda_{11}^{-1}\Lambda_{12}(\mathbb E[z_2]-\mu_2)$.
The fixed point $m=\mu$ solves both equations (unique when $\Lambda$ is positive definite): the means are exact.
The variances $1/\Lambda_{11}$ are the **conditional** variances $\Sigma_{11}-\Sigma_{12}^2/\Sigma_{22}$, smaller than the marginal $\Sigma_{11}$ whenever $z_1,z_2$ are correlated. For $\rho=0.9$, $1-\rho^2=0.19$ instead of 1.

</details>

## Part C · Code

### Exercise 10 · CAVI for a correlated Gaussian
*💻 Code · ★★☆*

Implement the updates from Exercise 9 for $\mu=(1,-1)$, $\Sigma=\begin{pmatrix}1&0.9\\0.9&1\end{pmatrix}$, starting from $m=(-3,3)$, 100 sweeps.
Store the final means `m_cavi`, the $q$ variances `v_cavi`, and plot the true density's contour with the $q$ contour on top.

In [ ]:
mu_t = np.array([1.0, -1.0]); Sig_t = np.array([[1, 0.9], [0.9, 1]]); Lam_t = np.linalg.inv(Sig_t)
m_cavi = None
v_cavi = None

check('means are exact', m_cavi, mu_t, tol=1e-4)
check('variances = 1/Lambda_ii (conditional variances)', v_cavi, 1 / np.diag(np.linalg.inv(Sig_t)))

<details>
<summary><b>💡 Hint</b></summary>

Update $m_1$ using the current $m_2$, then $m_2$ using the **new** $m_1$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The means converge geometrically (rate $\rho^2=0.81$ per sweep) to $(1,-1)$, while the variances are $0.19$ instead of $1$: the $q$ ellipse is a small circle inside the long diagonal ellipse of $p$, a textbook picture of VI's overconfidence.

```python
mu_t = np.array([1.0, -1.0]); Sig_t = np.array([[1, 0.9], [0.9, 1]]); Lam_t = np.linalg.inv(Sig_t)
m = np.array([-3.0, 3.0])
for _ in range(100):
    m[0] = mu_t[0] - Lam_t[0, 1] / Lam_t[0, 0] * (m[1] - mu_t[1])
    m[1] = mu_t[1] - Lam_t[1, 0] / Lam_t[1, 1] * (m[0] - mu_t[0])
m_cavi, v_cavi = m, 1 / np.diag(Lam_t)
g1, g2 = np.meshgrid(np.linspace(-2, 4, 100), np.linspace(-4, 2, 100)); G = np.dstack([g1, g2])
plt.contour(g1, g2, stats.multivariate_normal(mu_t, Sig_t).pdf(G), colors='k')
plt.contour(g1, g2, stats.multivariate_normal(m_cavi, np.diag(v_cavi)).pdf(G), colors='r')
plt.gca().set_aspect('equal'); plt.title('p (black) vs mean-field q (red)'); plt.show()
```

</details>

### Exercise 11 · Reverse vs forward KL on a bimodal target
*💻 Code · ★★☆*

Target $p=0.5\,\mathcal N(-3,1)+0.5\,\mathcal N(3,1)$. Fit a single Gaussian $q=\mathcal N(m,s^2)$ by
(a) minimizing the **reverse** KL $D_{KL}(q\Vert p)$ with a grid search over $m\in[-5,5]$, $s\in[0.3,4]$ (numerical integration on a fine $z$ grid), and
(b) minimizing the **forward** KL $D_{KL}(p\Vert q)$, whose minimizer is moment matching. Store `(m_rev, s_rev)` and `(m_fwd, s_fwd)`.

In [ ]:
zg = np.linspace(-10, 10, 4001)
p_bi = 0.5 * stats.norm.pdf(zg, -3, 1) + 0.5 * stats.norm.pdf(zg, 3, 1)
m_rev = s_rev = None
m_fwd = s_fwd = None

check('reverse KL locks onto one mode', None if m_rev is None else (abs(abs(m_rev) - 3) < 0.15 and abs(s_rev - 1) < 0.1), True)
check('forward KL = moment matching (mean)', m_fwd, 0.0, tol=1e-3)
check('forward KL = moment matching (std)', s_fwd, np.sqrt(1 + 9), tol=1e-3)

<details>
<summary><b>💡 Hint 1</b></summary>

Integrate with `np.trapezoid(q * (np.log(q) - np.log(p)), zg)` (or `np.trapz` in older NumPy); add a tiny constant inside the logs.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Moment matching: $m=\mathbb E_p[z]$, $s^2=\mathrm{Var}_p[z]=\mathbb E[\mathrm{Var}]+\mathrm{Var}[\mathbb E]$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $m\approx\pm3$, $s\approx1$: $q$ picks one mode and ignores the other (two equally good solutions by symmetry). Reverse KL $\approx\ln2\approx0.69$, the price of ignoring half the mass.
(b) $m=0$, $s=\sqrt{10}\approx3.16$: $q$ covers both modes but puts its peak where $p$ has almost no mass.
Neither is "right"; they answer different questions. VI uses (a) because its expectations are under $q$, which we can sample.

```python
zg = np.linspace(-10, 10, 4001)
p_bi = 0.5 * stats.norm.pdf(zg, -3, 1) + 0.5 * stats.norm.pdf(zg, 3, 1)
_trap = getattr(np, 'trapezoid', None) or np.trapz
best = (np.inf, None, None)
for m in np.linspace(-5, 5, 101):
    for s in np.linspace(0.3, 4, 75):
        q = stats.norm.pdf(zg, m, s)
        kl = _trap(q * (np.log(q + 1e-300) - np.log(p_bi + 1e-300)), zg)
        if kl < best[0]:
            best = (kl, m, s)
_, m_rev, s_rev = best
m_fwd = _trap(zg * p_bi, zg)
s_fwd = np.sqrt(_trap((zg - m_fwd) ** 2 * p_bi, zg))
print(best, m_fwd, s_fwd)
```

</details>

### Exercise 12 · A Monte Carlo ELBO
*💻 Code · ★☆☆*

For the shared conjugate model (`x_vi`, prior $\mathcal N(0,10^2)$, unit noise variance) implement `elbo_mc(m, s, n, rng)`: draw $z\sim q=\mathcal N(m,s^2)$ and
average $\log p(x,z)-\log q(z)$. Evaluate it (a) at the exact posterior and (b) at $q=\mathcal N(\text{post mean}+0.3,\ (2\cdot\text{post std})^2)$.
Store both in `elbo_exact`, `elbo_off`. Why does (a) have **zero** Monte Carlo variance?

In [ ]:
def log_joint(z):
    # log p(x, z) for an array of z values
    return stats.norm.logpdf(z, 0, np.sqrt(prior_var)) + stats.norm.logpdf(x_vi[:, None], np.atleast_1d(z), 1).sum(0)

def elbo_mc(m, s, n, rng):
    # TODO
    return None

elbo_exact = elbo_mc(post_mean_exact, np.sqrt(post_var_exact), 1000, np.random.default_rng(0))
elbo_off = elbo_mc(post_mean_exact + 0.3, 2 * np.sqrt(post_var_exact), 1000, np.random.default_rng(0))

_logev = stats.multivariate_normal(np.zeros(N_vi), np.eye(N_vi) + prior_var * np.ones((N_vi, N_vi))).logpdf(x_vi)
check('ELBO at the exact posterior = log evidence', elbo_exact, _logev, tol=1e-6)
check('ELBO is lower for a worse q', None if elbo_off is None else elbo_off < _logev - 1, True)

<details>
<summary><b>💡 Hint</b></summary>

`z = rng.normal(m, s, n)`; `np.mean(log_joint(z) - stats.norm.logpdf(z, m, s))`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

At the exact posterior $\log p(x,z)-\log q(z)=\log p(x,z)-\log p(z\mid x)=\log p(x)$ for **every** $z$: the integrand is constant, so even a single sample gives the exact log evidence.
The worse $q$ gives an ELBO about 3 nats below; the difference is exactly $D_{KL}(q\Vert p(z\mid x))$. The check computes $\log p(x)$ independently from $x\sim\mathcal N(0,I+100\,\mathbf 1\mathbf 1^\top)$.

```python
def log_joint(z):
    return stats.norm.logpdf(z, 0, np.sqrt(prior_var)) + stats.norm.logpdf(x_vi[:, None], np.atleast_1d(z), 1).sum(0)

def elbo_mc(m, s, n, rng):
    z = rng.normal(m, s, n)
    return np.mean(log_joint(z) - stats.norm.logpdf(z, m, s))

elbo_exact = elbo_mc(post_mean_exact, np.sqrt(post_var_exact), 1000, np.random.default_rng(0))
elbo_off = elbo_mc(post_mean_exact + 0.3, 2 * np.sqrt(post_var_exact), 1000, np.random.default_rng(0))
print(elbo_exact, elbo_off)
```

</details>

### Exercise 13 · Black-box VI with the reparameterization trick
*💻 Code · ★★★*

Fit $q=\mathcal N(m,e^{2\ell})$ to the shared posterior by stochastic gradient **ascent** on the ELBO with reparameterized gradients:
$z=m+e^\ell\varepsilon$, $\nabla_m\approx\frac1S\sum g(z_s)$, $\nabla_\ell\approx\frac1S\sum g(z_s)\varepsilon_se^\ell+1$, where $g(z)=\partial_z\log p(x,z)=-z/100+\sum_i(x_i-z)$
and the $+1$ is the gradient of the Gaussian entropy. Use $S=10$, learning rate $2\cdot10^{-3}$, 4 000 steps, and average the last 1 000 iterates.
Store `m_bbvi` and `s_bbvi`.

In [ ]:
m_bbvi = None
s_bbvi = None

check('mean', m_bbvi, post_mean_exact, tol=0.01)
check('std', s_bbvi, np.sqrt(post_var_exact), tol=0.005)

<details>
<summary><b>💡 Hint 1</b></summary>

Start at $m=0$, $\ell=0$. Use a seeded generator for $\varepsilon$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Iterate averaging (Polyak) removes most of the noise of the stochastic updates.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Both parameters land on the exact posterior ($m\approx2.13$, $s\approx0.141$) because the true posterior is in the Gaussian family. Nothing model-specific was needed except $\partial_z\log p(x,z)$, which autodiff provides in Pyro/NumPyro/Stan's ADVI ([[Tools and Libraries]]).
With mini-batches of data (scale the likelihood gradient by $N/B$) this becomes stochastic VI and scales to huge data sets.

```python
_r = np.random.default_rng(1)
m, l, lr, hist = 0.0, 0.0, 2e-3, []
for t in range(4000):
    eps = _r.normal(size=10)
    z = m + np.exp(l) * eps
    g = -z / prior_var + (x_vi.sum() - N_vi * z)
    m += lr * g.mean()
    l += lr * ((g * eps * np.exp(l)).mean() + 1)
    if t >= 3000:
        hist.append((m, np.exp(l)))
m_bbvi, s_bbvi = np.mean(hist, 0)
print(m_bbvi, s_bbvi, '| exact', post_mean_exact, np.sqrt(post_var_exact))
```

</details>

### Exercise 14 · Score-function vs reparameterization gradients
*💻 Code · ★★★*

For $q=\mathcal N(\mu,\sigma^2)$ with $\mu=1.5,\sigma=0.5$ and $f(z)=z^2$, estimate $\nabla_\mu\mathbb E_q[f]$ (true value 3) with single-sample estimators:
the **score-function / REINFORCE** estimator $f(z)\,\nabla_\mu\log q(z)=f(z)\frac{z-\mu}{\sigma^2}$ and the **reparameterization** estimator $f'(\mu+\sigma\varepsilon)$.
Using 100 000 draws, store the mean and variance of each (`sf_mean, sf_var, rp_mean, rp_var`).

In [ ]:
sf_mean = sf_var = rp_mean = rp_var = None

check('score-function is unbiased', sf_mean, 3.0, tol=0.05)
check('reparameterization is unbiased', rp_mean, 3.0, tol=0.02)
check('reparameterization has much lower variance', None if rp_var is None else rp_var * 5 < sf_var, True)

<details>
<summary><b>💡 Hint</b></summary>

Draw $\varepsilon$ once and use $z=\mu+\sigma\varepsilon$ for both estimators.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Both are unbiased, but the score-function variance is about $55$ vs about $1$ for the reparameterization estimator. The score function only uses the value $f(z)$ and the direction $(z-\mu)$, while reparameterization uses the derivative $f'$.
Score-function estimators still matter because they work for discrete latent variables and non-differentiable $f$; control variates (baselines) reduce their variance.

```python
_e = np.random.default_rng(3).normal(size=100000)
z = 1.5 + 0.5 * _e
sf = z ** 2 * (z - 1.5) / 0.5 ** 2
rp = 2 * z
sf_mean, sf_var, rp_mean, rp_var = sf.mean(), sf.var(), rp.mean(), rp.var()
print(sf_mean, sf_var, rp_mean, rp_var)
```

</details>

### Exercise 15 · CAVI for a Gaussian with unknown mean and precision
*💻 Code · ★★★*

The classic example (Bishop §10.1.3): $x_i\sim\mathcal N(\mu,\tau^{-1})$, prior $\mu\mid\tau\sim\mathcal N(\mu_0,(\lambda_0\tau)^{-1})$, $\tau\sim\mathrm{Gamma}(a_0,b_0)$,
mean-field $q(\mu)q(\tau)$ with $q(\mu)=\mathcal N(\mu_N,\lambda_N^{-1})$, $q(\tau)=\mathrm{Gamma}(a_N,b_N)$:
$$\mu_N=\tfrac{\lambda_0\mu_0+N\bar x}{\lambda_0+N},\quad\lambda_N=(\lambda_0+N)\,\mathbb E[\tau],\quad a_N=a_0+\tfrac{N+1}2,$$
$$b_N=b_0+\tfrac12\,\mathbb E_\mu\Big[\textstyle\sum_i(x_i-\mu)^2+\lambda_0(\mu-\mu_0)^2\Big].$$
Run CAVI on the data below ($\mu_0=0,\lambda_0=1,a_0=b_0=1$) and compare $\mathbb E_q[\tau]$ and $\mathrm{Var}_q[\mu]$ with the exact Normal–Gamma posterior. Store `E_tau_vi`, `var_mu_vi`.

In [ ]:
x_ng = np.random.default_rng(7).normal(1.0, 0.5, 10)
mu0, lam0, a0, b0 = 0.0, 1.0, 1.0, 1.0
E_tau_vi = None
var_mu_vi = None

_N, _xb = len(x_ng), x_ng.mean()
_a = a0 + _N / 2; _b = b0 + 0.5 * np.sum((x_ng - _xb) ** 2) + lam0 * _N * (_xb - mu0) ** 2 / (2 * (lam0 + _N))
check('E[tau] close to exact', None if E_tau_vi is None else abs(E_tau_vi / (_a / _b) - 1) < 0.1, True)
check('Var[mu] underestimated vs exact Student-t', None if var_mu_vi is None else var_mu_vi < _b / (_a - 1) / (lam0 + _N), True)

<details>
<summary><b>💡 Hint</b></summary>

$\mathbb E_\mu[(x_i-\mu)^2]=(x_i-\mu_N)^2+1/\lambda_N$. Iterate the two updates (q(μ), then q(τ)) ~50 times.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\mu_N$ does not depend on $q(\tau)$, so it is exact after one step; $\mathbb E[\tau]$ and $\lambda_N$ converge within a few sweeps.
$\mathbb E_q[\tau]$ is close to the exact $a/b$, but $\mathrm{Var}_q[\mu]=1/\lambda_N$ is smaller than the exact posterior variance of $\mu$, a Student-$t$ that accounts for the uncertainty in $\tau$. Mean field ignores the dependence of $\mu$'s spread on $\tau$: underestimated variance again.

```python
x_ng = np.random.default_rng(7).normal(1.0, 0.5, 10)
mu0, lam0, a0, b0 = 0.0, 1.0, 1.0, 1.0
N, xb = len(x_ng), x_ng.mean()
muN = (lam0 * mu0 + N * xb) / (lam0 + N)
aN = a0 + (N + 1) / 2
E_tau = a0 / b0
for _ in range(50):
    lamN = (lam0 + N) * E_tau
    bN = b0 + 0.5 * (np.sum((x_ng - muN) ** 2) + N / lamN + lam0 * ((muN - mu0) ** 2 + 1 / lamN))
    E_tau = aN / bN
E_tau_vi, var_mu_vi = E_tau, 1 / lamN
print(E_tau_vi, var_mu_vi)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Variational Inference](Variational%20Inference.md).*